# LLM Architecture Inspector

A **compact, architecture-agnostic notebook** to inspect Hugging Face models:

- What class / architecture is it?
- How many parameters (total & trainable)?
- How are parameters distributed across modules?
- Simple tree view of the submodule hierarchy
- Optional bar chart of largest modules
- Optional dummy forward pass to inspect output shapes

Works with **any `transformers` model** that can be loaded via `AutoModel`.

Requirements:
- `transformers`
- `torch`
- (optional) `matplotlib` for visualization


In [ ]:
# Install dependencies if needed (uncomment if running in a fresh env)
# %pip install transformers torch matplotlib

import math
import torch
from transformers import AutoConfig, AutoModel, AutoTokenizer

torch.set_grad_enabled(False)

# --- Change this to inspect a different model ---
MODEL_NAME = "gpt2"  # e.g., "meta-llama/Llama-3-8B-Instruct", local path, etc.

print(f"Loading model: {MODEL_NAME}")
config = AutoConfig.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, config=config)
model.eval().cpu()

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception:
    tokenizer = None
    print("[Info] No tokenizer available (non-text model or not provided). Forward demo will be limited.")

## 1. Basic Model & Config Info

This cell prints high-level architecture details pulled from the model class and config.

In [ ]:
print("=== BASIC INFO ===")
print(f"Model name:       {MODEL_NAME}")
print(f"Model class:      {model.__class__.__name__}")
print(f"Config class:     {type(config).__name__}")
print(f"Model type:       {getattr(config, 'model_type', None)}")

key_attrs = [
    "architectures",
    "hidden_size",
    "d_model",
    "num_hidden_layers",
    "num_attention_heads",
    "intermediate_size",
    "vocab_size",
    "max_position_embeddings",
]

for attr in key_attrs:
    print(f"{attr:22s}: {getattr(config, attr, None)}")

## 2. Parameter Counts

Total parameters, trainable vs frozen, and per-module stats.

In [ ]:
def count_parameters(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(model)

print("=== PARAMETER COUNTS ===")
print(f"Total parameters:        {total_params:,}  (~{total_params/1e6:.2f}M)")
print(f"Trainable parameters:    {trainable_params:,}  (~{trainable_params/1e6:.2f}M)")
print(f"Frozen parameters:       {total_params - trainable_params:,}  (~{(total_params-trainable_params)/1e6:.2f}M)\n")

# Per-module parameter statistics (non-recursive per module)
per_module = []
for name, module in model.named_modules():
    params_here = sum(p.numel() for p in module.parameters(recurse=False))
    if params_here > 0:
        per_module.append((name or "[root]", module.__class__.__name__, params_here))

per_module_sorted = sorted(per_module, key=lambda x: x[2], reverse=True)

print("Top 20 modules by *local* parameter count (no children):")
for name, cls_name, n in per_module_sorted[:20]:
    print(f"- {name:40s} | {cls_name:25s} | {n:,} (~{n/1e6:.3f}M)")

## 3. Architecture Tree View

Indented tree of the module hierarchy, with **cumulative** parameters per node (including children).

In [ ]:
def module_param_count(module):
    return sum(p.numel() for p in module.parameters())

def print_tree(module, name="[model]", indent=""):
    params = module_param_count(module)
    print(f"{indent}{name} ({module.__class__.__name__}) - {params/1e6:.2f}M params")
    for child_name, child in module.named_children():
        child_label = child_name
        print_tree(child, child_label, indent + "  ")

print("=== MODULE TREE (cumulative params) ===")
print_tree(model)

## 4. Simple Visualization (Optional)

Bar chart of the largest modules by local parameter count. Skips silently if `matplotlib` is not installed.

In [ ]:
try:
    import matplotlib.pyplot as plt

    if not per_module_sorted:
        print("No module stats computed.")
    else:
        top_n = 20
        top = per_module_sorted[:top_n]
        labels = [n for (n, _, _) in top]
        values = [p / 1e6 for (_, _, p) in top]  # in millions

        plt.figure(figsize=(8, max(4, 0.4 * len(labels))))
        y = range(len(labels))
        plt.barh(y, values)
        plt.yticks(y, labels)
        plt.xlabel("Parameters (millions)")
        plt.title("Top modules by local parameter count")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
except ImportError:
    print("matplotlib not installed; skipping visualization.")

## 5. Dummy Forward Pass (Shapes Only)

Runs a small forward pass **if a tokenizer is available** and prints output tensor shapes.

This is intentionally generic and may show slightly different keys depending on the architecture
(encoder-only, decoder-only, encoder-decoder, etc.).

In [ ]:
if tokenizer is None:
    print("No tokenizer; skipping forward pass demo.")
else:
    text = "Hello, this is a quick test to inspect output shapes."
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)

    print("=== FORWARD PASS OUTPUT SHAPES ===")
    if isinstance(outputs, dict):
        for k, v in outputs.items():
            try:
                print(f"{k:20s}: shape={tuple(v.shape)}")
            except AttributeError:
                print(f"{k:20s}: type={type(v)}")
    else:
        try:
            print("outputs shape:", tuple(outputs.shape))
        except AttributeError:
            print("outputs type:", type(outputs))

## 6. Compact JSON Summary (Optional)

Creates a small Python dict with key metrics that you can log or save.

In [ ]:
summary = {
    "model_name": MODEL_NAME,
    "model_class": model.__class__.__name__,
    "config_class": type(config).__name__,
    "model_type": getattr(config, "model_type", None),
    "num_parameters": int(total_params),
    "num_trainable_parameters": int(trainable_params),
    "hidden_size": getattr(config, "hidden_size", getattr(config, "d_model", None)),
    "num_hidden_layers": getattr(config, "num_hidden_layers", None),
    "num_attention_heads": getattr(config, "num_attention_heads", None),
    "intermediate_size": getattr(config, "intermediate_size", None),
    "vocab_size": getattr(config, "vocab_size", None),
}

print("=== SUMMARY DICT ===")
for k, v in summary.items():
    print(f"{k:25s}: {v}")

# Optionally save to JSON
# import json
# with open(f"{MODEL_NAME.replace('/', '_')}_summary.json", "w") as f:
#     json.dump(summary, f, indent=2)
# print("Saved summary JSON.")